# Sequential CSV Dimension Fill — GPU Only, Windows-Safe Checkpoints

This notebook fills dimensions using **only** `dim_backprop_gpu_only.py` and its CuPy/CUDA finite-field Jacobian-rank computation.

The search is driven by the MFAs already recorded in each CSV:

1. Read rows whose `is_minimal` value is true.

2. Group those minimal filling architectures by depth `h` and exponent.

3. Build the finite rectangular search box whose coordinatewise upper corner is the coordinatewise maximum of those MFAs.

4. Process candidates from small to large.

5. For each candidate:

   - if it is a **strict coordinatewise predecessor of at least one recorded MFA**, compute its dimension unless a valid row already exists;

   - otherwise skip it without calling the dimension oracle.

6. Save new results through retrying, resumable checkpoints.

On Windows, Excel, OneDrive, antivirus software, or another Python process may temporarily lock the destination CSV. If replacement remains blocked after several retries, the notebook writes the complete current data to `NAME.locked_checkpoint.csv` and continues instead of losing the GPU run. A later run automatically resumes from that checkpoint when it is newer than the primary CSV.


## Configuration

Place this notebook, `dim_backprop_gpu_only.py`, and the `Data/` directory in the same project folder. The module loader also recognizes common alternate filenames, including the uploaded filename with `(2)` appended.


Note: This is kind of ugly. Can probably rewrite this.

In [249]:
from pathlib import Path

# Paths
DATA_DIR = Path("Data")
FALLBACK_DATA_DIRS = [Path("data/raw"), Path("../data/raw")]

# None means every *_architectures.csv file in DATA_DIR.
# Example: CSV_FILES = [Path("Data/2_2_architectures.csv")]
CSV_FILES = None

# Set an explicit module path only when automatic discovery is undesirable.
GPU_MODULE_PATH = None
GPU_MODULE_FILENAMES = ["dim_backprop_gpu_only.py"]

# Search filters
# None means use every (h, exponent) group represented by an is_minimal=True row.
H_VALUES = None
EXPONENTS = None
DEFAULT_EXPONENT = 2

# Hidden widths begin at 1. A candidate must be strictly coordinatewise below at least one marked MFA before it is eligible for a dimension computation.
MIN_HIDDEN_WIDTH = 1

# The rectangular list can be inspected before running. Set a finite guard if desired.
MAX_SEARCH_BOX_SIZE = None

# GPU dimension oracle
PRIMES = (10_000_019, 19_511_957) # prefer to pick larger primes and multiple primes
SEED = 20260630
RANK_WORKSPACE_BYTES = 512 * 1024**2
DIMENSION_VERBOSE = False

# Execution controls
RUN_FILL = True
MAX_NEW_EVALUATIONS_PER_FILE = None # Change this to limit number of evaluations needed to be performed.

# 1 preserves the old crash-safe behavior. Increasing this reduces CSV write overhead.
SAVE_EVERY_N_NEW_ROWS = 1_000 
PROGRESS_EVERY = 1_000

# Windows may temporarily deny os.replace() when the target CSV is open in
# Excel, synchronized by OneDrive, scanned by antivirus, or used elsewhere.
SAVE_REPLACE_RETRIES = 8
SAVE_RETRY_DELAY_SECONDS = 1.0
SAVE_LOCKED_CHECKPOINT_SUFFIX = ".locked_checkpoint"
CONTINUE_WITH_LOCKED_CHECKPOINT = True

# A full-dimensional strict predecessor contradicts the recorded MFA flag.
# The row is saved first, then the notebook stops when this is True.
STOP_ON_MFA_CONTRADICTION = True


## Load the GPU-only dimension module

In [250]:
import ast
import importlib.util
import itertools
import math
import os
import sys
import time
import uuid
from collections import defaultdict
from typing import Iterable, Iterator, Sequence
import pandas as pd

In [251]:
def discover_gpu_module_path() -> Path:
    if GPU_MODULE_PATH is not None:
        path = Path(GPU_MODULE_PATH).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"GPU_MODULE_PATH does not exist: {path}")
        return path

    roots = [Path.cwd(), Path.cwd().parent, Path("/mnt/data")]
    checked = []
    for root in roots:
        for filename in GPU_MODULE_FILENAMES:
            candidate = (root / filename).resolve()
            checked.append(candidate)
            if candidate.exists():
                return candidate

    checked_text = "\n".join(f"  - {path}" for path in checked)
    raise FileNotFoundError(
        "Could not find the GPU-only dimension module. Checked:\n" + checked_text
    )

In [252]:
def load_gpu_dimension_module(path: Path):
    module_name = "_mfa_dim_backprop_gpu_only"
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create an import specification for {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

In [253]:
GPU_MODULE_FILE = discover_gpu_module_path()
gpu_dimension_module = load_gpu_dimension_module(GPU_MODULE_FILE)
compute_dimension = gpu_dimension_module.compute_dimension
gpu_information = gpu_dimension_module.gpu_information

print("GPU dimension module:", GPU_MODULE_FILE)
print("GPU information:", gpu_information())


GPU dimension module: C:\Users\daoke\Documents\GitHub\MFA_PNNs\notebooks\dim_backprop_gpu_only.py
GPU information: {'device_id': 0, 'name': 'NVIDIA GeForce RTX 4070 SUPER', 'device_count': 1, 'cupy_version': '14.1.1', 'cuda_runtime_version': 12090, 'driver_version': 12060}


## CSV and architecture helpers

In [254]:
def choose_data_dir() -> Path:
    if DATA_DIR.exists():
        return DATA_DIR
    for candidate in FALLBACK_DATA_DIRS:
        if candidate.exists():
            print(f"DATA_DIR={DATA_DIR!s} was not found. Using {candidate!s}.")
            return candidate
    return DATA_DIR


def find_csv_files(data_dir: Path) -> list[Path]:
    if CSV_FILES is not None:
        return [Path(path) for path in CSV_FILES]
    return sorted(data_dir.glob("*_architectures.csv"))



def locked_checkpoint_path(path: Path) -> Path:
    """Return the stable fallback path used when Windows locks the main CSV."""
    path = Path(path)
    return path.with_name(f"{path.stem}{SAVE_LOCKED_CHECKPOINT_SUFFIX}{path.suffix}")


def preferred_csv_source(path: Path) -> Path:
    """Prefer a newer locked-file checkpoint so interrupted runs resume safely."""
    path = Path(path)
    checkpoint = locked_checkpoint_path(path)

    if checkpoint.exists() and (
        not path.exists() or checkpoint.stat().st_mtime_ns > path.stat().st_mtime_ns
    ):
        print(
            f"Resuming from newer checkpoint {checkpoint.name!r} because "
            f"{path.name!r} was previously locked."
        )
        return checkpoint

    return path

In [255]:
def read_architecture_csv(path: Path) -> pd.DataFrame:
    source = preferred_csv_source(path)
    if source.exists():
        try:
            df = pd.read_csv(source)
        except Exception:
            # Compatibility with older processed files containing a preamble line.
            df = pd.read_csv(source, skiprows=1)
    else:
        df = pd.DataFrame()

    required_columns = [
        "h",
        "exponent",
        "architecture",
        "num_parameters",
        "dimension_computed",
        "ambient_dimension",
        "is_full_dimension",
        "is_minimal",
    ]
    audit_columns = [
        "expected_dimension",
        "defect_expected",
        "defect_ambient",
        "backend",
        "primes",
        "elapsed_seconds",
        "status",
        "covered_by_mfa",
    ]
    for column in required_columns + audit_columns:
        if column not in df.columns:
            df[column] = pd.Series(dtype="object")
    return df

def parse_architecture(value) -> tuple[int, ...]:
    if isinstance(value, (tuple, list)):
        architecture = tuple(int(x) for x in value)
    else:
        if pd.isna(value):
            raise ValueError("missing architecture")
        architecture = tuple(int(x) for x in ast.literal_eval(str(value)))
    if len(architecture) < 2 or any(width <= 0 for width in architecture):
        raise ValueError(f"invalid architecture: {architecture}")
    return architecture


def architecture_string(architecture: Sequence[int]) -> str:
    return str([int(width) for width in architecture])


def truthy(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y", "t"}


def infer_d0_dL_from_filename(path: Path) -> tuple[int, int]:
    parts = path.stem.split("_")
    if len(parts) >= 3 and parts[-1] == "architectures":
        return int(parts[0]), int(parts[1])
    raise ValueError(
        f"Could not infer d0 and dL from {path.name!r}; expected a name such as "
        "2_1_architectures.csv."
    )


def parameter_count(architecture: Sequence[int]) -> int:
    return sum(
        int(input_width) * int(output_width)
        for input_width, output_width in zip(architecture[:-1], architecture[1:])
    )


def hidden_leq(left: Sequence[int], right: Sequence[int]) -> bool:
    return len(left) == len(right) and all(int(a) <= int(b) for a, b in zip(left, right))


def full_architecture(d0: int, hidden: Sequence[int], dL: int) -> tuple[int, ...]:
    return (int(d0), *(int(width) for width in hidden), int(dL))


def stable_seed(
    base_seed: int,
    hidden: Sequence[int],
    h: int,
    d0: int,
    dL: int,
    exponent: int,
) -> int:
    value = (
        int(base_seed)
        + 99_991 * int(h)
        + 101 * int(d0)
        + 103 * int(dL)
        + 107 * int(exponent)
    )
    for index, width in enumerate(hidden):
        value += (index + 1) * 1_000_003 * int(width)
    return value % (2**31 - 1)


def valid_dimension_row(row) -> bool:
    try:
        parse_architecture(row["architecture"])
        dimension = row.get("dimension_computed")
        ambient = row.get("ambient_dimension")
        if pd.isna(dimension) or pd.isna(ambient):
            return False
        int(dimension)
        int(ambient)
        return True
    except Exception:
        return False


def architecture_exponent_key(
    architecture: Sequence[int] | str,
    exponent: int,
) -> tuple[str, int]:
    if isinstance(architecture, str):
        architecture_key = architecture_string(parse_architecture(architecture))
    else:
        architecture_key = architecture_string(architecture)
    return architecture_key, int(exponent)


def existing_rows_by_key(df: pd.DataFrame) -> dict[tuple[str, int], int]:
    index_by_key: dict[tuple[str, int], int] = {}
    for index, row in df.iterrows():
        try:
            exponent_value = row.get("exponent")
            exponent = DEFAULT_EXPONENT if pd.isna(exponent_value) else int(exponent_value)
            key = architecture_exponent_key(
                parse_architecture(row["architecture"]),
                exponent,
            )
            index_by_key[key] = index
        except Exception:
            continue
    return index_by_key



def save_csv(df: pd.DataFrame, path: Path) -> Path:
    """Save atomically when possible and survive Windows destination locks.

    Returns the path that actually received the complete CSV. Normally this is
    ``path``. If Windows keeps the destination locked, it is the stable
    ``*.locked_checkpoint.csv`` fallback instead.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = locked_checkpoint_path(path)
    temporary = path.with_name(
        f".{path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp"
    )

    try:
        # Write a complete independent file before touching the existing CSV.
        df.to_csv(temporary, index=False)

        last_permission_error = None
        retries = max(1, int(SAVE_REPLACE_RETRIES))
        for attempt in range(1, retries + 1):
            try:
                os.replace(temporary, path)

                # A successful primary save supersedes any stale fallback.
                if checkpoint.exists():
                    try:
                        checkpoint.unlink()
                    except OSError:
                        pass
                return path

            except PermissionError as exc:
                last_permission_error = exc
                if attempt < retries:
                    print(
                        f"CSV is locked: {path}. Retrying save "
                        f"({attempt}/{retries}) ...",
                        flush=True,
                    )
                    time.sleep(max(0.0, float(SAVE_RETRY_DELAY_SECONDS)))

        if not CONTINUE_WITH_LOCKED_CHECKPOINT:
            raise last_permission_error

        # Keep one predictable latest checkpoint. If that checkpoint is itself
        # open, use a unique recovery name rather than losing the in-memory rows.
        try:
            os.replace(temporary, checkpoint)
            recovery_path = checkpoint
        except PermissionError:
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            recovery_path = path.with_name(
                f"{path.stem}.locked_checkpoint_{timestamp}_{uuid.uuid4().hex[:8]}"
                f"{path.suffix}"
            )
            os.replace(temporary, recovery_path)

        print(
            "WARNING: Windows would not allow replacement of the primary CSV. "
            f"The complete current data was saved to:\n  {recovery_path}\n"
            "Close the primary CSV in Excel or any other program. A later "
            "checkpoint will automatically restore normal saving when the lock "
            "is released.",
            flush=True,
        )
        return recovery_path

    finally:
        # Remove only a leftover temporary file; never remove a recovery file.
        if temporary.exists():
            try:
                temporary.unlink()
            except OSError:
                pass


## Read the recorded MFAs and build the candidate list

Only rows already marked `is_minimal=True` define the search. The notebook does not discover or update the MFA list while it runs.

In [256]:
def selected(value: int, allowed: Sequence[int] | None) -> bool:
    return allowed is None or int(value) in {int(x) for x in allowed}


def recorded_mfa_groups(
    df: pd.DataFrame,
    d0: int,
    dL: int,
) -> dict[tuple[int, int], list[tuple[int, ...]]]:
    """Return {(h, exponent): [hidden MFA tuples]} from is_minimal=True rows."""
    groups: dict[tuple[int, int], list[tuple[int, ...]]] = defaultdict(list)

    for index, row in df.iterrows():
        if not truthy(row.get("is_minimal")):
            continue

        architecture = parse_architecture(row["architecture"])
        if architecture[0] != d0 or architecture[-1] != dL:
            raise ValueError(
                f"Row {index} is marked minimal but has endpoints {architecture[0], architecture[-1]}, "
                f"whereas {d0, dL} were inferred from the filename."
            )

        h = len(architecture) - 1
        if not pd.isna(row.get("h")) and int(row["h"]) != h:
            raise ValueError(
                f"Row {index} has h={row['h']} but architecture {architecture} has h={h}."
            )

        exponent_value = row.get("exponent")
        exponent = DEFAULT_EXPONENT if pd.isna(exponent_value) else int(exponent_value)
        if not selected(h, H_VALUES) or not selected(exponent, EXPONENTS):
            continue

        if not truthy(row.get("is_full_dimension")):
            print(
                f"Warning: row {index} is marked is_minimal=True but is_full_dimension is not true. "
                "It will still be used because is_minimal is the requested source of truth."
            )

        groups[(h, exponent)].append(tuple(architecture[1:-1]))

    cleaned: dict[tuple[int, int], list[tuple[int, ...]]] = {}
    for key, hidden_values in groups.items():
        unique = sorted(set(hidden_values))

        # Recorded MFAs should form an antichain. Equality was removed above.
        for i, left in enumerate(unique):
            for j, right in enumerate(unique):
                if i != j and hidden_leq(left, right):
                    raise ValueError(
                        f"The is_minimal rows for group {key} are not an antichain: "
                        f"{left} <= {right}. Fix the CSV flags before running."
                    )
        cleaned[key] = unique

    return dict(sorted(cleaned.items()))


def coordinatewise_mfa_maxima(mfas: Sequence[Sequence[int]]) -> tuple[int, ...]:
    if not mfas:
        raise ValueError("at least one MFA is required")
    hidden_length = len(mfas[0])
    if any(len(mfa) != hidden_length for mfa in mfas):
        raise ValueError("all MFAs in a group must have the same number of hidden layers")
    return tuple(max(int(mfa[i]) for mfa in mfas) for i in range(hidden_length))


def is_strict_predecessor_of_some_mfa(
    hidden: Sequence[int],
    mfas: Sequence[Sequence[int]],
) -> bool:
    hidden_tuple = tuple(int(width) for width in hidden)
    return any(
        hidden_tuple != tuple(int(width) for width in mfa)
        and hidden_leq(hidden_tuple, mfa)
        for mfa in mfas
    )


def covering_mfas(
    hidden: Sequence[int],
    mfas: Sequence[Sequence[int]],
) -> list[tuple[int, ...]]:
    hidden_tuple = tuple(int(width) for width in hidden)
    return [
        tuple(int(width) for width in mfa)
        for mfa in mfas
        if hidden_tuple != tuple(int(width) for width in mfa)
        and hidden_leq(hidden_tuple, mfa)
    ]


def candidate_priority(hidden: tuple[int, ...], d0: int, dL: int) -> tuple:
    architecture = full_architecture(d0, hidden, dL)
    return (parameter_count(architecture), sum(hidden), max(hidden), hidden)


def enumerate_search_box(
    mfas: Sequence[Sequence[int]],
    d0: int,
    dL: int,
) -> tuple[list[tuple[int, ...]], tuple[int, ...]]:
    """Materialize and sort the MFA-determined rectangular search box."""
    maxima = coordinatewise_mfa_maxima(mfas)
    if any(maximum < MIN_HIDDEN_WIDTH for maximum in maxima):
        raise ValueError(f"MFA maxima {maxima} lie below MIN_HIDDEN_WIDTH={MIN_HIDDEN_WIDTH}")

    box_size = math.prod(maximum - MIN_HIDDEN_WIDTH + 1 for maximum in maxima)
    if MAX_SEARCH_BOX_SIZE is not None and box_size > int(MAX_SEARCH_BOX_SIZE):
        raise RuntimeError(
            f"The MFA-determined search box contains {box_size:,} candidates, exceeding "
            f"MAX_SEARCH_BOX_SIZE={int(MAX_SEARCH_BOX_SIZE):,}."
        )

    ranges = [range(MIN_HIDDEN_WIDTH, maximum + 1) for maximum in maxima]
    candidates = [tuple(values) for values in itertools.product(*ranges)]
    candidates.sort(key=lambda hidden: candidate_priority(hidden, d0, dL))
    return candidates, maxima


## GPU evaluation and CSV updates

In [257]:
def record_for_architecture(
    architecture: Sequence[int],
    exponent: int,
    seed: int,
    covering: Sequence[Sequence[int]],
) -> dict:
    start = time.perf_counter()
    result = compute_dimension(
        architecture,
        exponent,
        primes=PRIMES,
        seed=seed,
        rank_workspace_bytes=RANK_WORKSPACE_BYTES,
        verbose=DIMENSION_VERBOSE,
    )
    elapsed = time.perf_counter() - start

    sizes, returned_exponent, ambient, expected, dimension, expected_defect = result
    if tuple(int(width) for width in sizes) != tuple(int(width) for width in architecture):
        raise RuntimeError(f"Dimension module returned unexpected architecture {sizes}")
    if int(returned_exponent) != int(exponent):
        raise RuntimeError(f"Dimension module returned unexpected exponent {returned_exponent}")

    is_full = int(dimension) == int(ambient)
    return {
        "h": len(architecture) - 1,
        "exponent": int(exponent),
        "architecture": architecture_string(architecture),
        "num_parameters": parameter_count(architecture),
        "dimension_computed": int(dimension),
        "ambient_dimension": int(ambient),
        "is_full_dimension": bool(is_full),
        # The notebook never promotes new rows into the recorded MFA list.
        "is_minimal": False,
        "expected_dimension": int(expected),
        "defect_expected": int(expected_defect),
        "defect_ambient": int(ambient) - int(dimension),
        "backend": "dim_backprop_gpu_only/CuPy-CUDA",
        "primes": str(tuple(int(prime) for prime in PRIMES)),
        "elapsed_seconds": elapsed,
        "status": "filling_below_recorded_mfa" if is_full else "nonfilling",
        "covered_by_mfa": str([list(map(int, mfa)) for mfa in covering]),
    }


def append_or_update_row(
    df: pd.DataFrame,
    record: dict,
    index_by_key: dict[tuple[str, int], int],
) -> tuple[pd.DataFrame, dict[tuple[str, int], int]]:
    row_key = architecture_exponent_key(record["architecture"], record["exponent"])

    for column in record:
        if column not in df.columns:
            df[column] = pd.Series(dtype="object")

    if row_key in index_by_key:
        index = index_by_key[row_key]
        for column, value in record.items():
            df.at[index, column] = value
    else:
        index = len(df)
        df.loc[index, list(record.keys())] = list(record.values())
        index_by_key[row_key] = index

    return df, index_by_key


def fill_one_csv_file(
    path: Path,
    *,
    max_new_evaluations: int | None = None, reverse = False
) -> dict:
    print("=" * 100)
    print(f"Processing {path}")
    print("=" * 100)

    df = read_architecture_csv(path)
    d0, dL = infer_d0_dL_from_filename(path)
    groups = recorded_mfa_groups(df, d0, dL)
    index_by_key = existing_rows_by_key(df)

    if not groups:
        print("No selected is_minimal=True rows were found. Nothing to evaluate.")
        return {
            "path": str(path),
            "new_evaluations": 0,
            "existing_rows": 0,
            "skipped_not_below_mfa": 0,
            "groups": [],
        }

    total_new = 0
    total_existing = 0
    total_skipped_not_below = 0
    total_marked_mfas = 0
    group_summaries = []
    unsaved_new = 0

    for (h, exponent), mfas in groups.items():
        candidates, maxima = enumerate_search_box(mfas, d0, dL)
        print(f"\nGroup h={h}, exponent={exponent}")
        print("Recorded MFA hidden tuples:", mfas)
        print("Coordinatewise search-box maximum:", maxima)
        print(f"Enumerated candidates: {len(candidates):,}")

        group_new = 0
        group_existing = 0
        group_skipped = 0
        group_marked_mfas = 0
        group_eligible = 0
        group_start = time.perf_counter()
        mfa_set = set(mfas)

        if reverse == True:
            candidates = list(reversed(candidates))
            
        for position, hidden in enumerate(candidates, start=1):
            if max_new_evaluations is not None and total_new >= int(max_new_evaluations):
                print(
                    f"Stopping {path.name} after "
                    f"max_new_evaluations={int(max_new_evaluations)}."
                )
                break

            architecture = full_architecture(d0, hidden, dL)
            row_key = architecture_exponent_key(architecture, exponent)

            # The recorded MFA rows themselves are boundary data, not targets.
            if hidden in mfa_set:
                group_marked_mfas += 1
                total_marked_mfas += 1
                continue

            # Requested fast gate: evaluate only when the candidate is below an MFA.
            covering = covering_mfas(hidden, mfas)
            if not covering:
                group_skipped += 1
                total_skipped_not_below += 1
                continue

            group_eligible += 1

            if row_key in index_by_key:
                row = df.loc[index_by_key[row_key]]
                if valid_dimension_row(row):
                    group_existing += 1
                    total_existing += 1
                    continue

            seed = stable_seed(SEED, hidden, h, d0, dL, exponent)
            print(
                f"[{position:,}/{len(candidates):,}] Evaluating {architecture} "
                f"below {len(covering)} MFA(s) ...",
                flush=True,
            )
            record = record_for_architecture(architecture, exponent, seed, covering)
            print(
                f"  {record['status'].upper()} | "
                f"dim={record['dimension_computed']}/{record['ambient_dimension']} | "
                f"params={record['num_parameters']} | "
                f"elapsed={record['elapsed_seconds']:.2f}s"
            )

            df, index_by_key = append_or_update_row(
                df,
                record,
                index_by_key,
            )
            group_new += 1
            total_new += 1
            unsaved_new += 1

            should_checkpoint = (
                SAVE_EVERY_N_NEW_ROWS is not None
                and int(SAVE_EVERY_N_NEW_ROWS) > 0
                and unsaved_new >= int(SAVE_EVERY_N_NEW_ROWS)
            )
            if should_checkpoint:
                save_csv(df, path)
                unsaved_new = 0

            if record["is_full_dimension"]:
                # Save before stopping: the contradiction is useful diagnostic data.
                save_csv(df, path)
                unsaved_new = 0
                message = (
                    f"Architecture {architecture} is a strict predecessor of recorded MFA(s) "
                    f"{covering}, but the GPU computation found full ambient dimension. "
                    "The CSV's is_minimal flags are inconsistent with this result."
                )
                if STOP_ON_MFA_CONTRADICTION:
                    raise RuntimeError(message)
                print("WARNING:", message)

            if PROGRESS_EVERY and group_new % int(PROGRESS_EVERY) == 0:
                elapsed = time.perf_counter() - group_start
                print(
                    f"Progress h={h}, exponent={exponent}: new={group_new}, "
                    f"existing={group_existing}, skipped={group_skipped}, "
                    f"elapsed={elapsed:.1f}s"
                )

        if unsaved_new:
            save_csv(df, path)
            unsaved_new = 0

        elapsed = time.perf_counter() - group_start
        summary = {
            "h": h,
            "exponent": exponent,
            "mfas": list(mfas),
            "search_box_maxima": maxima,
            "enumerated": len(candidates),
            "eligible_below_mfa": group_eligible,
            "new_evaluations": group_new,
            "existing_rows": group_existing,
            "marked_mfas_skipped": group_marked_mfas,
            "not_below_any_mfa_skipped": group_skipped,
            "elapsed_seconds": elapsed,
        }
        group_summaries.append(summary)
        print(
            f"Finished h={h}, exponent={exponent}: eligible={group_eligible}, "
            f"new={group_new}, existing={group_existing}, "
            f"MFA rows={group_marked_mfas}, not-below skips={group_skipped}, "
            f"elapsed={elapsed:.1f}s"
        )

        if max_new_evaluations is not None and total_new >= int(max_new_evaluations):
            break

    save_csv(df, path)
    return {
        "path": str(path),
        "d0": d0,
        "dL": dL,
        "new_evaluations": total_new,
        "existing_rows": total_existing,
        "marked_mfas_skipped": total_marked_mfas,
        "skipped_not_below_mfa": total_skipped_not_below,
        "groups": group_summaries,
    }


## Inspect the planned files and MFA-derived search sizes

In [258]:
data_dir = choose_data_dir()
csv_paths = find_csv_files(data_dir)

print("Data directory:", data_dir.resolve())
print("CSV files:")
for path in csv_paths:
    print(" -", path)

if not csv_paths:
    raise FileNotFoundError(
        f"No *_architectures.csv files found in {data_dir!s}. "
        "Set DATA_DIR or CSV_FILES in the configuration cell."
    )

plan_rows = []
for path in csv_paths:
    df = read_architecture_csv(path)
    d0, dL = infer_d0_dL_from_filename(path)
    groups = recorded_mfa_groups(df, d0, dL)
    for (h, exponent), mfas in groups.items():
        maxima = coordinatewise_mfa_maxima(mfas)
        box_size = math.prod(maximum - MIN_HIDDEN_WIDTH + 1 for maximum in maxima)
        plan_rows.append({
            "file": str(path),
            "h": h,
            "exponent": exponent,
            "number_of_mfas": len(mfas),
            "mfas": mfas,
            "box_maxima": maxima,
            "box_size": box_size,
        })

plan_df = pd.DataFrame(plan_rows)
display(plan_df)


DATA_DIR=Data was not found. Using ..\data\raw.
Data directory: C:\Users\daoke\Documents\GitHub\MFA_PNNs\data\raw
CSV files:
 - ..\data\raw\1_1_r1_architectures.csv
 - ..\data\raw\1_1_r2_architectures.csv
 - ..\data\raw\1_1_r3_architectures.csv
 - ..\data\raw\1_1_r4_architectures.csv
 - ..\data\raw\1_1_r5_architectures.csv
 - ..\data\raw\1_1_r6_architectures.csv
 - ..\data\raw\1_1_r7_architectures.csv
 - ..\data\raw\1_1_r8_architectures.csv
 - ..\data\raw\1_1_r9_architectures.csv
 - ..\data\raw\1_2_r1_architectures.csv
 - ..\data\raw\1_2_r2_architectures.csv
 - ..\data\raw\1_2_r3_architectures.csv
 - ..\data\raw\1_2_r4_architectures.csv
 - ..\data\raw\1_2_r5_architectures.csv
 - ..\data\raw\1_2_r6_architectures.csv
 - ..\data\raw\1_2_r7_architectures.csv
 - ..\data\raw\1_2_r8_architectures.csv
 - ..\data\raw\1_2_r9_architectures.csv
 - ..\data\raw\1_3_r1_architectures.csv
 - ..\data\raw\1_3_r2_architectures.csv
 - ..\data\raw\1_3_r3_architectures.csv
 - ..\data\raw\1_3_r4_architectures

,file,h,exponent,number_of_mfas,mfas,box_maxima,box_size
0,..\data\raw\1_1_r1_architectures.csv,2,1,1,"[(1,)]","(1,)",1
1,..\data\raw\1_1_r1_architectures.csv,3,1,1,"[(1, 1)]","(1, 1)",1
2,..\data\raw\1_1_r1_architectures.csv,4,1,1,"[(1, 1, 1)]","(1, 1, 1)",1
3,..\data\raw\1_1_r1_architectures.csv,5,1,1,"[(1, 1, 1, 1)]","(1, 1, 1, 1)",1
4,..\data\raw\1_1_r1_architectures.csv,6,1,1,"[(1, 1, 1, 1, 1)]","(1, 1, 1, 1, 1)",1
...,...,...,...,...,...,...,...
746,..\data\raw\9_1_r2_architectures.csv,2,2,1,"[(9,)]","(9,)",9
747,..\data\raw\9_1_r2_architectures.csv,3,2,4,"[(25, 18), (26, 16), (27, 14), (28, 13)]","(28, 18)",504
748,..\data\raw\9_1_r3_architectures.csv,2,3,1,"[(19,)]","(19,)",19
749,..\data\raw\9_1_r4_architectures.csv,2,4,1,"[(55,)]","(55,)",55


## Run the sequential fill

The notebook checkpoints to the same CSV. Rerunning it skips every architecture whose row already contains a valid computed and ambient dimension.

### Select Particular CSV_Files

In [259]:
# csv_paths=csv_paths[100:]
csv_paths

[WindowsPath('../data/raw/1_1_r1_architectures.csv'),
 WindowsPath('../data/raw/1_1_r2_architectures.csv'),
 WindowsPath('../data/raw/1_1_r3_architectures.csv'),
 WindowsPath('../data/raw/1_1_r4_architectures.csv'),
 WindowsPath('../data/raw/1_1_r5_architectures.csv'),
 WindowsPath('../data/raw/1_1_r6_architectures.csv'),
 WindowsPath('../data/raw/1_1_r7_architectures.csv'),
 WindowsPath('../data/raw/1_1_r8_architectures.csv'),
 WindowsPath('../data/raw/1_1_r9_architectures.csv'),
 WindowsPath('../data/raw/1_2_r1_architectures.csv'),
 WindowsPath('../data/raw/1_2_r2_architectures.csv'),
 WindowsPath('../data/raw/1_2_r3_architectures.csv'),
 WindowsPath('../data/raw/1_2_r4_architectures.csv'),
 WindowsPath('../data/raw/1_2_r5_architectures.csv'),
 WindowsPath('../data/raw/1_2_r6_architectures.csv'),
 WindowsPath('../data/raw/1_2_r7_architectures.csv'),
 WindowsPath('../data/raw/1_2_r8_architectures.csv'),
 WindowsPath('../data/raw/1_2_r9_architectures.csv'),
 WindowsPath('../data/raw/1_

In [260]:
all_summaries = []

if RUN_FILL:
    for csv_path in csv_paths:
        summary = fill_one_csv_file(csv_path, max_new_evaluations=MAX_NEW_EVALUATIONS_PER_FILE, reverse=True)
        all_summaries.append(summary)
    print("\nAll requested CSV files are complete for the selected MFA predecessor boxes.")
else:
    print("RUN_FILL is False. Review plan_df, then set RUN_FILL=True to begin.")

all_summaries


Processing ..\data\raw\1_1_r1_architectures.csv

Group h=2, exponent=1
Recorded MFA hidden tuples: [(1,)]
Coordinatewise search-box maximum: (1,)
Enumerated candidates: 1
Finished h=2, exponent=1: eligible=0, new=0, existing=0, MFA rows=1, not-below skips=0, elapsed=0.0s

Group h=3, exponent=1
Recorded MFA hidden tuples: [(1, 1)]
Coordinatewise search-box maximum: (1, 1)
Enumerated candidates: 1
Finished h=3, exponent=1: eligible=0, new=0, existing=0, MFA rows=1, not-below skips=0, elapsed=0.0s

Group h=4, exponent=1
Recorded MFA hidden tuples: [(1, 1, 1)]
Coordinatewise search-box maximum: (1, 1, 1)
Enumerated candidates: 1
Finished h=4, exponent=1: eligible=0, new=0, existing=0, MFA rows=1, not-below skips=0, elapsed=0.0s

Group h=5, exponent=1
Recorded MFA hidden tuples: [(1, 1, 1, 1)]
Coordinatewise search-box maximum: (1, 1, 1, 1)
Enumerated candidates: 1
Finished h=5, exponent=1: eligible=0, new=0, existing=0, MFA rows=1, not-below skips=0, elapsed=0.0s

Group h=6, exponent=1
Rec

KeyboardInterrupt: 

## Post-run summary

In [261]:
for path in csv_paths:
    df = read_architecture_csv(path)
    print("=" * 100)
    print(path)
    print("rows:", len(df))
    print("is_full_dimension counts:")
    print(df["is_full_dimension"].value_counts(dropna=False))
    print("status counts:")
    print(df["status"].value_counts(dropna=False).head(20))
    marked = df[df["is_minimal"].map(truthy)]
    print("Recorded MFAs:")
    display(marked[["h", "exponent", "architecture", "dimension_computed", "ambient_dimension", "num_parameters",]])


..\data\raw\1_1_r1_architectures.csv
rows: 62
is_full_dimension counts:
is_full_dimension
True    62
Name: count, dtype: int64
status counts:
status
NaN                           61
filling_below_recorded_mfa     1
Name: count, dtype: int64
Recorded MFAs:


,h,exponent,architecture,dimension_computed,ambient_dimension,num_parameters
50,2.0,1.0,"[1, 1, 1]",1.0,1.0,2.0
58,3.0,1.0,"[1, 1, 1, 1]",1.0,1.0,3.0
59,4.0,1.0,"[1, 1, 1, 1, 1]",1.0,1.0,4.0
60,5.0,1.0,"[1, 1, 1, 1, 1, 1]",1.0,1.0,5.0
61,6.0,1.0,"[1, 1, 1, 1, 1, 1, 1]",1.0,1.0,6.0


..\data\raw\1_1_r2_architectures.csv
rows: 72
is_full_dimension counts:
is_full_dimension
True    72
Name: count, dtype: int64
status counts:
status
NaN    72
Name: count, dtype: int64
Recorded MFAs:


,h,exponent,architecture,dimension_computed,ambient_dimension,num_parameters
48,2,2,"[1, 1, 1]",1,1,2
65,3,2,"[1, 1, 1, 1]",1,1,3
66,4,2,"[1, 1, 1, 1, 1]",1,1,4
67,5,2,"[1, 1, 1, 1, 1, 1]",1,1,5
68,6,2,"[1, 1, 1, 1, 1, 1, 1]",1,1,6
69,7,2,"[1, 1, 1, 1, 1, 1, 1, 1]",1,1,7
70,8,2,"[1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1,8
71,9,2,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1,1,9


..\data\raw\1_1_r3_architectures.csv
rows: 52
is_full_dimension counts:
is_full_dimension
True    52
Name: count, dtype: int64
status counts:
status
NaN    52
Name: count, dtype: int64
Recorded MFAs:


...

,h,exponent,architecture,dimension_computed,ambient_dimension,num_parameters
2,2,2,"[6, 6, 1]",21,21,42


..\data\raw\6_1_r3_architectures.csv
rows: 177
is_full_dimension counts:
is_full_dimension
True     94
False    83
Name: count, dtype: int64
status counts:
status
NaN    177
Name: count, dtype: int64
Recorded MFAs:


,h,exponent,architecture,dimension_computed,ambient_dimension,num_parameters
7,2,3,"[6, 10, 1]",56,56,70
24,3,3,"[6, 38, 48, 1]",2002,2002,2100
43,3,3,"[6, 23, 83, 1]",2002,2002,2130
70,3,3,"[6, 46, 39, 1]",2002,2002,2109
76,3,3,"[6, 45, 40, 1]",2002,2002,2110
79,3,3,"[6, 32, 58, 1]",2002,2002,2106
91,3,3,"[6, 25, 76, 1]",2002,2002,2126
105,3,3,"[6, 34, 54, 1]",2002,2002,2094
109,3,3,"[6, 37, 50, 1]",2002,2002,2122
120,3,3,"[6, 36, 51, 1]",2002,2002,2103


..\data\raw\6_8_r9_architectures.csv
rows: 0
is_full_dimension counts:
Series([], Name: count, dtype: int64)
status counts:
Series([], Name: count, dtype: int64)
Recorded MFAs:


KeyError: "None of [Index(['h', 'exponent', 'architecture', 'dimension_computed',\n       'ambient_dimension', 'num_parameters'],\n      dtype='str')] are in the [columns]"

## Post-Run entries with Codimension = 1

In [263]:
# for path in csv_paths:
#     df = read_architecture_csv(path)
#     df = df[df['dimension_computed'] == df['ambient_dimension']-1]
#     df = df[df['h']==2]
#     if len(df)>0:
#         display(df)

dfs = []
for path in csv_paths:
    df = read_architecture_csv(path)
    df = df[df['dimension_computed'] == df['ambient_dimension'] - 1]
    # df = df[df['h'] == 2]
    if len(df) > 0:
        dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
display(merged_df)

,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,expected_dimension,defect_expected,defect_ambient,backend,primes,elapsed_seconds,status,covered_by_mfa
0,3.0,12.0,"[2, 9, 15, 1]",168.0,144.0,145.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,12.0,"[2, 8, 17, 1]",169.0,144.0,145.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,12.0,"[2, 6, 23, 1]",173.0,144.0,145.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,2.0,"[2, 1, 1]",3.0,2.0,3.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7.0,2.0,"[2, 4, 3, 4, 5, 5, 10, 1]",137.0,64.0,65.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1327,3.0,3.0,"[7, 36, 133, 1]",5173.0,5004.0,5005.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1328,2.0,7.0,"[7, 245, 1]",1960.0,1715.0,1716.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1329,2.0,2.0,"[8, 7, 1]",63.0,35.0,36.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1330,3.0,2.0,"[8, 20, 14, 1]",454.0,329.0,330.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [265]:
merged_df.to_csv('../data/processed/codimension_one.csv')